In [1]:
!pip -q install pinecone sentence-transformers langchain langchain-google-genai pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.8 MB/s eta 0:00:00


In [2]:
import os
import time
import yaml
import numpy as np

from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [3]:
from google.colab import userdata
openai_key = userdata.get('OPENAI_API_KEY')
google_key = userdata.get('GOOGLE_API_KEY')
pinecone_api_key = userdata.get('PINECONE_API_KEY')
# os.environ["GOOGLE_API_KEY"] = google_key

### Step 1 - Get your LLM ready

In [4]:
llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash', api_key=google_key)
print(llm.__class__.__name__)

ChatGoogleGenerativeAI


### Step 2 - Access your Pinecone Index and Embedding model

In [5]:
pc = Pinecone(api_key=pinecone_api_key)

index_name = "vector-docs"
index = pc.Index(index_name)

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Step 3 - Editing the query (customize)
 - Multi Query Expansion
 - HyDE

In [6]:
# query_text = "How do I get a two wheeler loan?"
# def edit_query(query: str) -> str:
#     # keep simple for now; return as-is
#     return query

In [7]:
# HOME WORK - TRY the same with a list of keywords you generated in the Upserting part
# query_text = "How do I get a loan against property loan?"

def edit_query(query: str) -> str:
    prompt = f"""
Expand this user query for retrieval.
Add a few useful similar words.
Return only one short expanded query.

Query: {query}
"""
    return llm.invoke(prompt).content.strip()

def make_keywords(text, max_words=6):
    words = text.lower().replace("?", "").replace(",", "").replace(".", "").replace(')', '').replace('(', '').split()
    words = list(dict.fromkeys(words))   # remove duplicates, keep order
    return words

In [8]:
# edit_query(query_text)

### Step 4 - Retrieval

In [9]:
# # 1. Define the user's question
# def retrieve_docs(query_text, top_k = 5):
#   query_vector = model.encode([query_text])[0].tolist()
#   results = index.query(
#       vector=query_vector,
#       top_k=5,
#       include_metadata=True
#   )
#   response = []
#   # You may tweak here to get whatver you need
#   for match in results["matches"]:
#       response.append({
#           "id": match["id"],
#           "score": float(match["score"]),
#           "text": match["metadata"]["text"]
#       })
#   return response

In [10]:
def retrieve_docs(query_text, keywords=None, top_k=5):
    query_vector = model.encode([query_text])[0].tolist()

    if keywords:
        results = index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True,
            filter={"keywords": {"$in": keywords}}
        )
    else:
        results = index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True
        )

    response = []
    for match in results["matches"]:
        response.append({
            "id": match["id"],
            "score": float(match["score"]),
            "text": match["metadata"]["text"]
            # "links":
        })
# Home work - Put links from metadata in the response
    return response

### Step 5 - Context building (customizations)
- Concatenate the context together
- Post filtering
- Add links
- shorten the context
- Summarize


In [11]:
def build_context(results, max_chars: int = 1200) -> str:
    parts = []
    total_chars = 0

    for item in results:
        chunk = item["text"].strip()

        remaining = max_chars - total_chars
        if remaining <= 0:
            break

        if len(chunk) > remaining:
            chunk = chunk[:remaining]

        parts.append(chunk)
        total_chars += len(chunk)

    return "\n\n".join(parts)

### Step 6 - Prompt Rendering and Augmentation (customizations)

In [12]:



def render_prompt(query: str, sources: str) -> str:
  PROMPT_TEMPLATE = """
Imagine you're a financial assitant, your job is to give grounded answers using only the Context below:
Make sure you don't answer from outside the given context, don't be robotic, and please answer in a friendly way.

Context:
{sources}

Question:
{query}
""".strip()
  return PROMPT_TEMPLATE.format(query=query, sources=sources)

## Step 7 - Making the chain

In [13]:
# See what's happening
def _start(query):
  return {"query":query}

debug_chain = (
    RunnableLambda(_start)
    .assign(edited_query=RunnableLambda(lambda d: edit_query(d["query"])))
    .assign(filter_keywords=RunnableLambda(lambda d: make_keywords(d["edited_query"], max_words=6)))
    .assign(results=RunnableLambda(lambda d: retrieve_docs(d["edited_query"], keywords=d["filter_keywords"], top_k=5)))
)

debug_output = debug_chain.invoke("How do I get a Loan Against Property?")
debug_output

{'query': 'How do I get a Loan Against Property?',
 'edited_query': 'How to get apply obtain a Loan Against Property LAP property-backed loan?',
 'filter_keywords': ['how',
  'to',
  'get',
  'apply',
  'obtain',
  'a',
  'loan',
  'against',
  'property',
  'lap',
  'property-backed'],
 'results': [{'id': 'loan_against_property.txt_chunk_0',
   'score': 0.613226175,
   'text': 'BrightBridge Finance — Loan Against Property (LAP)\nSimple loans. Clear terms. Fast decisions.\n\nOverview\nA Loan Against Property (LAP) is a secured loan where you pledge an owned property (residential or commercial, subject to policy) as collateral to borrow funds for a wide range of needs. LAP is often used for business expansion, education funding, medical expenses, or consolidating high-cost debt, while benefiting from longer tenors than typical unsecured loans.\n\nWhy LAP can be useful\n• Higher ticket sizes may be available because the loan is secured\n• Longer repayment tenure can reduce monthly burden

In [14]:
# See what's happening
def _start(query):
  return {"query":query}

final_chain = (
    RunnableLambda(_start)
    .assign(edited_query=RunnableLambda(lambda d: edit_query(d["query"])))
    .assign(filter_keywords=RunnableLambda(lambda d: make_keywords(d["edited_query"], max_words=6)))
    .assign(results=RunnableLambda(lambda d: retrieve_docs(d["edited_query"], keywords=d["filter_keywords"], top_k=5)))
    .assign(sources = RunnableLambda(lambda d: build_context(d['results'], max_chars = 1200))) # {'sources':All sources combined}
    .assign(prompt = RunnableLambda(lambda d: render_prompt(d['edited_query'], d['sources']))) # {'prompt': 'updated prompt'}
    .assign(answer = RunnableLambda(lambda d: llm.invoke(d['prompt'])))
    .pick('answer') | StrOutputParser()
)

final_output = final_chain.invoke("What documents are needed for Loan Against Property?")


In [15]:
print(final_output)

That's a great question! It's always smart to be prepared when applying for a loan.

However, after reviewing the information provided by BrightBridge Finance, the specific documents, paperwork, or a detailed checklist required for a Loan Against Property (LAP) are not mentioned. The context does advise applicants to "Ensure property documents are complete to avoid delays," but it doesn't list what those specific documents are.

So, while ensuring your property documents are in order is highlighted, the details of what those documents entail, or any other required paperwork or checklist, aren't available in this overview.
